# Apollo Audio Restoration - Kaggle Training Notebook

**Prerequisites:**
- Upload this repo as a Kaggle dataset (or clone it in a setup cell)
- Upload preprocessed MUSDB18-HQ HDF5 dataset to Kaggle
- GPU accelerator: T4 x1 (16GB VRAM)

Edit **Cell 1** to configure training, then run all cells.

In [ ]:
# ==============================================================
# Cell 1: Configuration - edit these values
# ==============================================================
BATCH_SIZE = 8          # T4 has 16GB; 8 is safe, 16 may OOM with Mamba
MAX_EPOCHS = 100        # Kaggle session limit is 12h; set accordingly
NUM_SAMPLES = 8000      # samples per epoch
NUM_WORKERS = 2         # Kaggle has limited CPU
DATASET_PATH = "/kaggle/input/musdb18-hq-apollo-preprocess"  # HDF5 dir
REPO_PATH = "/kaggle/input/apollo-upgrade"  # path to this repo dataset

# Mamba / model hyperparams (must match what you trained if resuming)
FEATURE_DIM = 256
LAYERS = 6
USE_MAMBA = 1
MAMBA_EXPAND = 2
MAMBA_DSTATE = 16
MAMBA_DCONV = 4
BANDWIDTH_D = 80
BANDWIDTH_N = 40
WIN_PARTS = 240

In [ ]:
# ==============================================================
# Cell 2: Install dependencies
# ==============================================================
!pip install -q pytorch-lightning hydra-core omegaconf h5py thop wandb soundfile \
    fast-bss-eval torch-mir-eval torchmetrics torch-optimizer huggingface-hub mamba-ssm torch-complex

In [ ]:
# ==============================================================
# Cell 3: Setup repo and working directory
# ==============================================================
import os
import shutil
import subprocess

WORK_DIR = "/kaggle/working/apollo"

# Copy repo to writable working directory
if not os.path.exists(WORK_DIR):
    shutil.copytree(REPO_PATH, WORK_DIR)
    print(f"Repo copied to {WORK_DIR}")
else:
    print(f"Repo already at {WORK_DIR}")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

In [ ]:
# ==============================================================
# Cell 4: Create Kaggle-specific training config
# ==============================================================
import yaml

config = {
    "exp": {"dir": "/kaggle/working/Exps", "name": "Apollo"},
    "seed": 614020,
    "datas": {
        "_target_": "look2hear.datas.MusdbMoisesdbDataModule",
        "train_dir": DATASET_PATH,
        "eval_dir": os.path.join(DATASET_PATH, "eval"),
        "codec_type": "mp3",
        "codec_options": {
            "bitrate": "random",
            "compression": "random",
            "complexity": "random",
            "vbr": "random",
        },
        "sr": 44100,
        "segments": 3,
        "num_stems": 8,
        "snr_range": [-10, 10],
        "num_samples": NUM_SAMPLES,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
    },
    "model": {
        "_target_": "look2hear.models.apollo.Apollo",
        "sr": 44100,
        "win": 20,
        "feature_dim": FEATURE_DIM,
        "layer": LAYERS,
    },
    "discriminator": {
        "_target_": "look2hear.discriminators.frequencydis.MultiFrequencyDiscriminator",
        "nch": 2,
        "window": [32, 64, 128, 256, 512, 1024, 2048],
    },
    "optimizer_g": {"_target_": "torch.optim.AdamW", "lr": 0.001, "weight_decay": 0.01},
    "optimizer_d": {
        "_target_": "torch.optim.AdamW",
        "lr": 0.0001,
        "weight_decay": 0.01,
        "betas": [0.5, 0.99],
    },
    "scheduler_g": {"_target_": "torch.optim.lr_scheduler.StepLR", "step_size": 2, "gamma": 0.98},
    "scheduler_d": {"_target_": "torch.optim.lr_scheduler.StepLR", "step_size": 2, "gamma": 0.98},
    "loss_g": {"_target_": "look2hear.losses.gan_losses.MultiFrequencyGenLoss", "eps": 1e-8},
    "loss_d": {"_target_": "look2hear.losses.gan_losses.MultiFrequencyDisLoss", "eps": 1e-8},
    "metrics": {"_target_": "look2hear.losses.MultiSrcNegSDR", "sdr_type": "sisdr"},
    "system": {"_target_": "look2hear.system.audio_litmodule.AudioLightningModule"},
    "early_stopping": {
        "_target_": "pytorch_lightning.callbacks.EarlyStopping",
        "monitor": "val_loss",
        "patience": 20,
        "mode": "min",
        "verbose": True,
    },
    "checkpoint": {
        "_target_": "pytorch_lightning.callbacks.ModelCheckpoint",
        "dirpath": "/kaggle/working/Exps/Apollo/checkpoints",
        "monitor": "val_loss",
        "mode": "min",
        "verbose": True,
        "save_top_k": 5,
        "save_last": True,
        "filename": "{epoch}-{val_loss:.4f}",
    },
    "logger": {
        "_target_": "pytorch_lightning.loggers.WandbLogger",
        "name": "Apollo",
        "save_dir": "/kaggle/working/Exps/Apollo/logs",
        "offline": True,  # no internet access during Kaggle training
        "project": "Audio-Restoration",
    },
    "trainer": {
        "_target_": "pytorch_lightning.Trainer",
        "devices": [0],  # single T4
        "max_epochs": MAX_EPOCHS,
        "sync_batchnorm": False,  # only needed for multi-GPU
        "default_root_dir": "/kaggle/working/Exps/Apollo/",
        "accelerator": "cuda",
        "precision": "16-mixed",  # fp16 mixed precision — halves VRAM usage
        "limit_train_batches": 1.0,
        "fast_dev_run": False,
    },
}

os.makedirs("/kaggle/working", exist_ok=True)
config_path = "/kaggle/working/kaggle_apollo.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config written to {config_path}")
print(f"  batch_size: {BATCH_SIZE}, max_epochs: {MAX_EPOCHS}, num_samples: {NUM_SAMPLES}")

In [ ]:
# ==============================================================
# Cell 5: Measure GMACs (random-init model, before training)
# ==============================================================
import os
import torch
import look2hear.models
from thop import profile

os.environ["A_USE_MAMBA"] = str(USE_MAMBA)
os.environ["A_FEATURE_DIM"] = str(FEATURE_DIM)
os.environ["A_LAYERS"] = str(LAYERS)
os.environ["A_BANDWIDTH_D"] = str(BANDWIDTH_D)
os.environ["A_BANDWIDTH_N"] = str(BANDWIDTH_N)
os.environ["A_WIN_PARTS"] = str(WIN_PARTS)
os.environ["A_MAMBA_EXPAND"] = str(MAMBA_EXPAND)
os.environ["A_MAMBA_DSTATE"] = str(MAMBA_DSTATE)
os.environ["A_MAMBA_DCONV"] = str(MAMBA_DCONV)

model = look2hear.models.Apollo(
    sr=44100,
    win=20,
    feature_dim=FEATURE_DIM,
    layer=LAYERS,
).cuda()

# 0.5s dummy input at 44.1kHz
dummy = torch.randn(1, 1, 22050).cuda()
macs, params = profile(model, inputs=(dummy,))
print(f"Parameters: {params:,.0f}")
print(f"GMACs (0.5s input): {macs / 1e9:.3f}")

del model
torch.cuda.empty_cache()

In [ ]:
# ==============================================================
# Cell 6: Train
# ==============================================================
import os
import subprocess

env = os.environ.copy()
env.update({
    "WANDB_MODE": "offline",
    "A_USE_MAMBA": str(USE_MAMBA),
    "A_FEATURE_DIM": str(FEATURE_DIM),
    "A_LAYERS": str(LAYERS),
    "A_BANDWIDTH_D": str(BANDWIDTH_D),
    "A_BANDWIDTH_N": str(BANDWIDTH_N),
    "A_WIN_PARTS": str(WIN_PARTS),
    "A_MAMBA_EXPAND": str(MAMBA_EXPAND),
    "A_MAMBA_DSTATE": str(MAMBA_DSTATE),
    "A_MAMBA_DCONV": str(MAMBA_DCONV),
})

cmd = ["python", "train.py", f"--conf_dir={config_path}"]
print("Starting training:", " ".join(cmd))
result = subprocess.run(cmd, env=env, cwd=WORK_DIR)
print(f"\nTraining exited with code: {result.returncode}")

In [ ]:
# ==============================================================
# Cell 7: Download checkpoint
# ==============================================================
import os
import shutil

src = "/kaggle/working/Exps/Apollo/best_model.pth"
dst = "/kaggle/working/best_model.pth"

if os.path.exists(src):
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f"Checkpoint saved to {dst} ({size_mb:.1f} MB)")
    print("Download it from the Kaggle output tab.")
else:
    print(f"ERROR: {src} not found. Training may have failed.")
    # List available checkpoints
    ckpt_dir = "/kaggle/working/Exps/Apollo/checkpoints"
    if os.path.exists(ckpt_dir):
        print("Available checkpoints:", os.listdir(ckpt_dir))

In [ ]:
# ==============================================================
# Cell 8: Inference
# ==============================================================
# Place your degraded wav at /kaggle/working/input.wav
# (upload via Kaggle file panel or copy from input dataset)
import os
import subprocess

INPUT_WAV = "/kaggle/working/input.wav"
OUTPUT_WAV = "/kaggle/working/restored.wav"
MODEL_PTH = "/kaggle/working/best_model.pth"

assert os.path.exists(INPUT_WAV), f"Input file not found: {INPUT_WAV}"
assert os.path.exists(MODEL_PTH), f"Model not found: {MODEL_PTH}"

env = os.environ.copy()
env.update({
    "A_USE_MAMBA": str(USE_MAMBA),
    "A_FEATURE_DIM": str(FEATURE_DIM),
    "A_LAYERS": str(LAYERS),
    "A_BANDWIDTH_D": str(BANDWIDTH_D),
    "A_BANDWIDTH_N": str(BANDWIDTH_N),
    "A_WIN_PARTS": str(WIN_PARTS),
    "A_MAMBA_EXPAND": str(MAMBA_EXPAND),
    "A_MAMBA_DSTATE": str(MAMBA_DSTATE),
    "A_MAMBA_DCONV": str(MAMBA_DCONV),
})

cmd = [
    "python", "inference.py",
    f"--in_wav={INPUT_WAV}",
    f"--out_wav={OUTPUT_WAV}",
    f"--model_path={MODEL_PTH}",
]
print("Running inference:", " ".join(cmd))
result = subprocess.run(cmd, env=env, cwd=WORK_DIR)
print(f"\nInference exited with code: {result.returncode}")
if os.path.exists(OUTPUT_WAV):
    print(f"Restored audio saved to {OUTPUT_WAV}")

In [ ]:
# ==============================================================
# Cell 9: SI-SDR Measurement
# ==============================================================
# Set CLEAN_WAV to the original (clean) reference, or leave None to skip baseline
CLEAN_WAV = None  # e.g. "/kaggle/working/clean.wav"

import torch
import torchaudio

def load_mono(path):
    """Load wav, mix to mono, return 1D tensor."""
    audio, sr = torchaudio.load(path)
    if audio.shape[0] > 1:
        audio = audio.mean(dim=0, keepdim=True)
    return audio.squeeze(0), sr

def si_sdr(reference, estimate):
    """Scale-invariant SDR in dB."""
    reference = reference - reference.mean()
    estimate = estimate - estimate.mean()
    dot = (reference * estimate).sum()
    s_target = dot / (reference ** 2).sum() * reference
    e_noise = estimate - s_target
    return 10 * torch.log10((s_target ** 2).sum() / (e_noise ** 2).sum()).item()

def trim_to_same(a, b):
    min_len = min(a.shape[-1], b.shape[-1])
    return a[..., :min_len], b[..., :min_len]

degraded, _ = load_mono(INPUT_WAV)
restored, _ = load_mono(OUTPUT_WAV)

if CLEAN_WAV is not None and os.path.exists(CLEAN_WAV):
    clean, _ = load_mono(CLEAN_WAV)
    clean_d, deg = trim_to_same(clean, degraded)
    clean_r, res = trim_to_same(clean, restored)

    baseline = si_sdr(clean_d, deg)
    improved = si_sdr(clean_r, res)
    print(f"SI-SDR (degraded):  {baseline:.2f} dB")
    print(f"SI-SDR (restored):  {improved:.2f} dB")
    print(f"SI-SDR improvement: {improved - baseline:+.2f} dB")
else:
    print("No clean reference provided. Skipping SI-SDR comparison.")
    print("Set CLEAN_WAV at the top of this cell to enable measurement.")
    d, r = trim_to_same(degraded, restored)
    print(f"\nSelf SI-SDR (restored vs degraded): {si_sdr(d, r):.2f} dB")